In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

df = pd.read_csv("/datasets/everpeak_retail.csv")
df_copy = df.copy()

print("--- Información General del Dataset ---")
print(df_copy.info())

print("\n--- Resumen Estadístico de Columnas Numéricas ---")
print(df_copy.describe())

--- Información General del Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5008 entries, 0 to 5007
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          5008 non-null   int64  
 1   order_date        5000 non-null   object 
 2   customer_id       5008 non-null   int64  
 3   product_category  5008 non-null   object 
 4   price             5008 non-null   int64  
 5   quantity          5008 non-null   int64  
 6   order_value       5008 non-null   int64  
 7   payment_method    5008 non-null   object 
 8   city              4908 non-null   object 
 9   state             4908 non-null   object 
 10  customer_age      4858 non-null   float64
dtypes: float64(1), int64(5), object(5)
memory usage: 430.5+ KB
None

--- Resumen Estadístico de Columnas Numéricas ---
          order_id  customer_id         price     quantity    order_value  \
count  5008.000000  5008.000000   5008.000000

In [2]:
# Análisis Exploratorio (EDA) y Diagnóstico
#Conteo inicial de ausencias
print("--- Conteo de Valores Faltantes por Columna ---")
print(df_copy.isna().sum())

# Validación de hipótesis de ausencia (Análisis de Datos Faltantes)
print("\n--- Relación de ausencias en 'city' según el método de pago (MCAR) ---")
missing_city_by_pay = (
    df_copy["city"].isna().groupby(df_copy["payment_method"]).mean() * 100
).sort_values(ascending=False)
print(missing_city_by_pay)

# Diagnóstico de Valores Atípicos / Sentinela en Edad
edades_sospechosas = df_copy[
    (df_copy["customer_age"] < 12) | (df_copy["customer_age"] > 100)
]
print("\n--- Diagnóstico de Valores Atípicos en 'customer_age' ---")
print(f"Total de registros fuera de rango común: {len(edades_sospechosas)}")
print(edades_sospechosas["customer_age"].value_counts())

--- Conteo de Valores Faltantes por Columna ---
order_id              0
order_date            8
customer_id           0
product_category      0
price                 0
quantity              0
order_value           0
payment_method        0
city                100
state               100
customer_age        150
dtype: int64

--- Relación de ausencias en 'city' según el método de pago (MCAR) ---
payment_method
credit_card    2.151714
debit_card     2.130045
cash           2.010050
paypal         1.531915
Name: city, dtype: float64

--- Diagnóstico de Valores Atípicos en 'customer_age' ---
Total de registros fuera de rango común: 25
-999.0    25
Name: customer_age, dtype: int64


In [3]:
def pipeline_limpieza_everpeak(df, median_fill_cols, fill_unknown_cols, date_drop_cols):
    """Pipeline profesional para la limpieza e imputación de datos.

    - Remueve valores centinela e imputa numéricos con la mediana.
    - Rellena categorías vacías con un identificador 'unknown'.
    - Transforma fechas y elimina registros inválidos.
    """
    df_local = df.copy()

    # Tratamiento de columnas numéricas y eliminación de valores basura
    for col in median_fill_cols:
        valores_basura = [-999, 999, 0, 1]
        df_local[col] = df_local[col].replace(valores_basura, np.nan)
        df_local[col] = pd.to_numeric(df_local[col], errors="coerce")

        # Calculamos la mediana limpia (sin el sesgo de los valores basura)
        mediana_limpia = df_local[col].median()
        df_local[col] = df_local[col].fillna(mediana_limpia)

    # Tratamiento de columnas categóricas (Estrategia MCAR)
    for col in fill_unknown_cols:
        df_local[col] = df_local[col].fillna("unknown")

    # Transformación de tipos de tiempo y manejo de nulos/fechas inválidas
    for col in date_drop_cols:
        df_local[col] = pd.to_datetime(df_local[col], errors="coerce")
        df_local = df_local.dropna(subset=[col]).reset_index(drop=True)

    return df_local

In [4]:
# Definición de listas de procesamiento basados en el diagnóstico
cols_mediana = ["customer_age"]
cols_unknown = ["city", "state"]
cols_fecha = ["order_date"]

# Ejecución del pipeline sobre nuestros datos analizados
df_final = pipeline_limpieza_everpeak(
    df_copy, cols_mediana, cols_unknown, cols_fecha
)

print("============ CONTROL DE CALIDAD FINAL ============")
print("\n--- Conteo Absoluto de Valores Nulos ---")
print(df_final.isna().sum())

print("\n--- Rango de Edades Verificado ---")
print(
    f"Edad Mínima Real: {df_final['customer_age'].min()} años | Edad Máxima Real: {df_final['customer_age'].max()} años"
)

print("\n--- Verificación del Tipo de Dato en Fechas ---")
print(f"Tipo de columna 'order_date': {df_final['order_date'].dtype}")

print("\n Dimensiones finales del dataset limpio:", df_final.shape)

============ CONTROL DE CALIDAD FINAL ============

--- Conteo Absoluto de Valores Nulos ---
order_id            0
order_date          0
customer_id         0
product_category    0
price               0
quantity            0
order_value         0
payment_method      0
city                0
state               0
customer_age        0
dtype: int64

--- Rango de Edades Verificado ---
Edad Mínima Real: 18.0 años | Edad Máxima Real: 80.0 años

--- Verificación del Tipo de Dato en Fechas ---
Tipo de columna 'order_date': datetime64[ns]

 Dimensiones finales del dataset limpio: (5000, 11)
